# P1-VL-30B-A3B — text problems

Runs [PRIME-RL/P1-VL-30B-A3B](https://huggingface.co/PRIME-RL/P1-VL-30B-A3B) on the text problems in `Problems/Text`
with the prompts from `Problems/Text/text_prompts.py`, and saves each answer to
`Result/finetune_models/Text/` as `<model>_<problem file>_<prompt>.md`, e.g. `P1-VL-30B-A3B_Problem_1_Obvious_Q_detail_system_prompt.md`.
For each answer, a metadata file with the same name is saved to `Result/finetune_models/Text/metadata/` (`requested_model`, `problem_question`, `prompt_used`, `total_output_tokens`, `total_time_taken` in seconds).

The model is loaded with `AutoModelForImageTextToText` (the auto class this model is registered under) and quantized to 4-bit with bitsandbytes
while loading. Answers are generated with `model.generate`.

**GPU:** needs an **80 GB GPU** (Colab: Runtime → Change runtime type → **H100**). bitsandbytes only quantizes the attention layers;
the experts (most of the model) stay in bf16, so the model takes ~56 GiB.

**On Colab:** Runtime → Run all. The first run downloads ~62 GB. Each answer is also copied to Google Drive
(`MyDrive/Physics_LLM_Test/Result/finetune_models/Text/`, metadata in its `metadata/` folder), and answers that already exist there are skipped, so after a disconnect just Run all again.

In [ ]:
## install the libraries (transformers version matches requirements.txt)
%pip install -q "transformers==5.17.0" accelerate bitsandbytes

In [ ]:
import sys
import json
import time
import subprocess
from pathlib import Path

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig, set_seed

In [ ]:
## check the GPU: the model needs ~60 GB of GPU memory

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. On Colab: Runtime -> Change runtime type -> H100.")

free, total = torch.cuda.mem_get_info()
print(f"GPU: {torch.cuda.get_device_name()}, {free / 2**30:.1f} GiB free of {total / 2**30:.1f} GiB")
if free / 2**30 < 60:
    raise RuntimeError("Not enough GPU memory. Use an 80 GB GPU (Colab: H100).")

In [ ]:
## find the repository: use it if this notebook is inside it, otherwise clone it (e.g. on Colab)

REPO_URL = "https://github.com/ShRuwali/Physics_LLM_Test.git"

if (Path.cwd().parent.parent / "Problems").exists():
    repo_dir = Path.cwd().parent.parent
else:
    repo_dir = Path.cwd() / "Physics_LLM_Test"
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)

print("Repository:", repo_dir)

In [ ]:
## use only 1 model

model_id = "PRIME-RL/P1-VL-30B-A3B"
model_name = model_id.split("/")[-1]

MAX_NEW_TOKENS = 16384   ## room for the model's reasoning plus the answer
SEED = 0                 ## same seed for every problem, so runs can be repeated

In [ ]:
# add the repository into path to import the prompts
sys.path.append(str(repo_dir))

from Problems.Text.text_prompts import prompts

print(prompts.keys())

In [ ]:
# Path to markdown files
problem_folder = repo_dir / "Problems" / "Text"

# List Markdown filenames, including the .md extension
file_names = sorted(file.name for file in problem_folder.glob("*.md"))

print("Total number of files:::", len(file_names))
print(file_names)

In [ ]:
## quantize the model to 4-bit with bitsandbytes while loading it
## lm_head and the vision encoder (model.visual) are kept in bf16 for accuracy

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["lm_head", "model.visual"],
)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

processor = AutoProcessor.from_pretrained(model_id)

print(f"Model loaded. GPU memory used: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

In [ ]:
## generate an answer with model.generate
## P1-VL is a reasoning model: it thinks first and writes the final answer after </think>.
## Only the final answer is saved (like the frontier model results).
## Also returns the number of generated tokens (reasoning + answer) and the time taken in seconds, for the metadata.

def generate_answer(messages):
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    set_seed(SEED)
    start_time = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)  ## sampling settings come from the model's generation_config

    total_time_taken = time.perf_counter() - start_time

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    total_output_tokens = len(new_tokens)
    output_text = processor.decode(new_tokens, skip_special_tokens=True)

    if "</think>" in output_text:
        answer = output_text.split("</think>", 1)[1].strip()
    else:
        answer = "[No final answer: the model reached MAX_NEW_TOKENS while still reasoning.]\n\n" + output_text.strip()
    return answer, total_output_tokens, total_time_taken

In [ ]:
## where to save the answers. On Colab each answer is also copied to Google Drive,
## so finished answers survive a disconnect and are skipped when you run again

output_folder = repo_dir / "Result" / "finetune_models" / "Text"
output_folder.mkdir(parents=True, exist_ok=True)
metadata_folder = output_folder / "metadata"
metadata_folder.mkdir(parents=True, exist_ok=True)

drive_folder = None
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_folder = Path("/content/drive/MyDrive/Physics_LLM_Test/Result/finetune_models/Text")
    (drive_folder / "metadata").mkdir(parents=True, exist_ok=True)


def already_done(output_filename):
    return any(folder is not None and (folder / output_filename).exists()
               for folder in (output_folder, drive_folder))


def save_answer(output_filename, output_text):
    for folder in (output_folder, drive_folder):
        if folder is not None:
            (folder / output_filename).write_text(output_text, encoding="utf-8")


## Create metadata: a json file with the same name as the answer, in the metadata folder

def save_metadata(output_filename, metadata):
    metadata_filename = Path(output_filename).with_suffix(".json").name
    for folder in (metadata_folder, drive_folder and drive_folder / "metadata"):
        if folder is not None:
            (folder / metadata_filename).write_text(
                json.dumps(metadata, indent=4, ensure_ascii=False), encoding="utf-8")

In [ ]:
%%time

for keys in prompts:
    for file in file_names:

        file_path = problem_folder / file

        if keys == "detail_system_prompt" and "regular" in file_path.stem.lower():  ## skip if keys have detail_system_prompt and
            continue     ## problems that have regular in their name. We dont need Regular problems to use detail prompts but run the rest

        output_filename = f"{model_name}_{file_path.stem}_{keys}.md"
        if already_done(output_filename):
            print(f"Already done, skipping:::{output_filename}")
            continue

        problem_text = file_path.read_text(encoding="utf-8")

        messages = [
            {"role": "system", "content": [{"type": "text", "text": prompts[keys]}]},
            {"role": "user", "content": [{"type": "text", "text": problem_text}]},
        ]

        output_text, total_output_tokens, total_time_taken = generate_answer(messages)
        save_answer(output_filename, output_text)

        ## Create metadata
        metadata = {
            "requested_model": model_id,
            "problem_question": file_path.stem,
            "prompt_used": prompts[keys],
            "total_output_tokens": total_output_tokens,
            "total_time_taken": total_time_taken,
        }
        save_metadata(output_filename, metadata)
        print(f"Completed:::{output_filename}")

In [ ]:
## list the saved answers
for file in sorted((drive_folder or output_folder).glob("*.md")):
    print(file.name)